# An Adjoint Formulation

In this example we will explore leveraging an adjoint formulation of the linear boltzmann equations for computing response functions in a desired region of interest (RoI) on a two dimensional grid.

This is a complete simulation transport example. Each aspect of this example can be broken as follows:
- [Mesh](#mesh)
- [Material IDs](#material-ids)
- [Point Source](#point-source)
- [Angular Quadrature](#angular-quadrature)
- [Linear Boltzmann Solver](#linear-boltzmann-solver)
- [A Region of Interest](#a-region-of-interest)
- [Adjoint Source](#adjoint-source)
- [Adjoint Solver](#adjoint-solver)
- [Evaluate Resposne](#evaluate-response)

## Using this Notebook
Before running this example, make sure that the **Python module of OpenSn** was installed.

### Converting and Running this Notebook from the Terminal
To run this notebook from the terminal, simply type:

`jupyter nbconvert --to python --execute first_example.ipynb`.

To run this notebook in parallel (for example, using 4 processes), simply type:

`mpiexec -n 4 jupyter nbconvert --to python --execute first_example.ipynb`.

In [ ]:
from mpi4py import MPI
size = MPI.COMM_WORLD.size
rank = MPI.COMM_WORLD.rank
barrier = MPI.COMM_WORLD.barrier

if rank == 0:
    print(f"Running the first LBS example with {size} MPI processors.")

## Import Requirements

Import required classes and functions from the Python interface of OpenSn. Make sure that the path
to PyOpenSn is appended to Python's PATH.

In [ ]:
# assuming that the execute dir is the notebook dir
# this line is not necessary when PyOpenSn is installed using pip
sys.path.append("../../../..")

from pyopensn.mesh import OrthogonalMeshGenerator
from pyopensn.xs import MultiGroupXS
from pyopensn.source import PointSource, VolumetricSource
from pyopensn.aquad import GLCProductQuadrature2DXY
from pyopensn.solver import DiscreteOrdinatesProblem, SteadyStateSolver
from pyopensn.response import ResponseEvaluator
from pyopensn.fieldfunc import FieldFunctionInterpolationVolume
from pyopensn.logvol import RPPLogicalVolume
from pyopensn.context import UseColor, Finalize


import os
import sys
import numpy as np

## Mesh
Here, we will use the in-house orthogonal mesh generator for a simple Cartesian grid.

### List of Nodes
We first create a list of nodes for each dimension in X and Y. Here, we mesh the X and Y directions with different node values to capture our RoI. 

For this, we will make use of a function `getNodes()`. Given material boundaries and the number of cells within each boundary the function returns a python list of node values. 

In [ ]:
def getNodes(n, dn, ref=1):
    '''
    n: list of material interfaces
    dn: number of cells for each region
    ref(default=1): refine the number of cells by an integer multiple
    '''
    dn = [i * ref for i in dn]
    nodes = []
    for i in range(len(n[:-1])):
        edges = np.linspace(n[i],n[i+1],dn[i]+1)
        if len(nodes) == 0:
            nodes = edges
        else:
            nodes = [*nodes, *edges[1:]]
    return list(nodes)

x = [0.0, 1.0, 2.0, 7.0, 8.0, 10.0]
dx = [2, 2, 10, 2, 4]
nodesx = getNodes(x, dx, ref=5)

y = [0.0, 1.0, 2.0, 4.5, 5.5, 10.0]
dy = [2, 2, 5, 2, 9]
nodesy = getNodes(y, dy, ref=5)

### Orthogonal Mesh Generation
We use the `OrthogonalMeshGenerator` and pass the list of nodes per dimension. Here, we pass our lsit of nodes for both the x and y directions creating a 2D geometry of side length 10 with square cells. 

In [ ]:
meshgen = OrthogonalMeshGenerator( node_sets=[nodesx, nodesy] )
grid = meshgen.Execute()

## Material IDs
When using the in-house `OrthogonalMeshGenerator`, no material IDs are assigned. The user needs to assign material IDs to all cells. We will begin by assigning the background material ID with a value of 0.

In [ ]:
background = RPPLogicalVolume(infx=True, infy=True, infz=True)
grid.SetBlockIDFromLogicalVolume(background, 0, True)

Next we will assign material IDs for the source and RoI. When assigning material IDs via logical volumes the IDs for each cell is overriden by the most recent assignment. That is to say, assigning a material ID of 1 to any region in our domain will override the previously set ID to those cells. 

In [ ]:
source = RPPLogicalVolume(
            xmin=1.0, xmax=2.0,
            ymin=1.0, ymax=2.0,
            infz=True
)
grid.SetBlockIDFromLogicalVolume(source, 1, True)

roi = RPPLogicalVolume(
            xmin=7.0, xmax=8.0,
            ymin=4.5, ymax=5.5,
            infz=True
)
grid.SetBlockIDFromLogicalVolume(roi, 2, True)

The resulting mesh and partition is shown below:
![Mesh_Partition](images/first_example_mesh_partition.png)

### Cross Sections
We create one-group cross sections using a built-in method. 
See the tutorials' section on cross sections for more details on how to load cross sections into OpenSn.

For this tutorial we will be using simple one-group cross sections. For the background we aim to replicate the properties of air by assigning a small interaction cross section and high scattering ratio. For the source and RoI $\sigma_t$ is made to be 1.0 where the source is a pure scatterer and the RoI a pure absorber. 

In [ ]:
xs_bkgrnd = MultiGroupXS()
xs_bkgrnd.CreateSimpleOneGroup(sigma_t=1e-4, c=0.9)

xs_src = MultiGroupXS()
xs_src.CreateSimpleOneGroup(sigma_t=1.0, c=1.0)

xs_roi = MultiGroupXS()
xs_roi.CreateSimpleOneGroup(sigma_t=1.0, c=0.0)

## Volumetric Source
We create a volumetric one-group source which will be assigned to cells with given block IDs.
Volumetric sources are assigned to the solver via the `options` parameter in the LBS block (see below).

In [ ]:
vol_src = VolumetricSource(block_ids=[1], group_strength=[1.0])

## Angular Quadrature
We create a product Gauss-Legendre-Chebyshev angular quadrature and pass the total number of polar cosines
(here `pol = 4`) and the number of azimuthal subdivisions in **four quadrants** (`azi = 64`).
This creates a 2D angular quadrature for XY geometry.

In [ ]:
pol = 4
azi = 64
pquad = GLCProductQuadrature2DXY(pol, azi)

## Linear Boltzmann Solver
### Options for the Linear Boltzmann Problem (LBS)
In the LBS block, we provide
+ the number of energy groups,
+ the groupsets (with 0-indexing), the handle for the angular quadrature, the angle aggregation, the solver type,
tolerances, and other solver options.

In [ ]:
phys = DiscreteOrdinatesProblem(
    mesh=grid,
    num_groups=1,
    groupsets=[
        {
            "groups_from_to": (0, 0),
            "angular_quadrature": pquad,
            "angle_aggregation_num_subsets": 1,
            "inner_linear_method": "petsc_gmres",
            "l_abs_tol": 1.0e-6,
            "l_max_its": 300,
            "gmres_restart_interval": 30
        }
    ],
    options={
        "volumetric_sources": [vol_src],
    },
    xs_map=[
        {"block_ids": [0], "xs": xs_bkgrnd},
        {"block_ids": [1], "xs": xs_src},
        {"block_ids": [2], "xs": xs_roi}
    ],
    options={
        "scattering_order": 0,
        "volumetric_sources": [vol_src],
    },
)

### Putting the Linear Boltzmann Solver Together
We then create the physics solver, initialize it, and execute it.

In [ ]:
ss_solver = SteadyStateSolver(lbs_problem=phys)
ss_solver.Initialize()
ss_solver.Execute()

## A Region of Interest

## Adjoint Source

## Adjoint Solver

## Evaluate Response

## Post-Processing via Field Functions
We extract the scalar flux (i.e., the first entry in the field function list; recall that lua
indexing starts at 1) and export it to a VTK file whose name is supplied by the user. See the tutorials' section
on post-processing for more details on field functions.

The resulting scalar flux is shown below:

![Scalar_flux](images/first_example_scalar_flux.png)

In [ ]:
fflist = phys.GetScalarFieldFunctionList(only_scalar_flux=False)
vtk_basename = "first_example"
FieldFunctionGridBased.ExportMultipleToVTK(
    [fflist[0][0]],  # export only the flux of group 0 (first []), moment 0 (second [])
    vtk_basename
)

## Finalize (for Jupyter Notebook only)

In Python script mode, PyOpenSn automatically handles environment termination. However, this
automatic finalization does not occur when running in a Jupyter notebook, so explicit finalization
of the environment at the end of the notebook is required. Do not call the finalization in Python
script mode, or in console mode.

Note that PyOpenSn's finalization must be called before MPI's finalization.


In [ ]:
from IPython import get_ipython

def finalize_env():
    Finalize()
    MPI.Finalize()

ipython_instance = get_ipython()
if ipython_instance is not None:
    ipython_instance.events.register("post_execute", finalize_env)

## Possible Extensions
1. Change the number of MPI processes;
2. Change the spatial resolution by increasing or decreasing the number of cells;
3. Change the angular resolution by increasing or decreasing the number of polar and azimuthal subdivisions.